Task 1.1 — Read the file the foundations way

In [1]:
import csv
import json

def parse_row(row):
    # Create a copy to avoid modifying the original row in-place
    parsed = dict(row)
    parsed['age'] = int(parsed['age'])
    parsed['fee_inr'] = int(parsed['fee_inr'])
    parsed['prior_no_shows'] = int(parsed['prior_no_shows'])
    parsed['distance_km'] = float(parsed['distance_km'])
    return parsed

accepted = []
rejected = []

with open('appointments.csv', mode='r', encoding='utf-8', newline='') as f:
    reader = csv.DictReader(f)
    header = reader.fieldnames
    print(f"Header: {header}")

    row_count = 0
    for row in reader:
        row_count += 1
        try:
            parsed_row = parse_row(row)
            accepted.append(parsed_row)
        except ValueError:
            rejected.append(row)

print(f"Total Row Count: {row_count}")
print(f"Accepted Rows: {len(accepted)}")
print(f"Rejected Rows: {len(rejected)}")

with open('rejected_rows.json', mode='w', encoding='utf-8') as f_out:
    json.dump(rejected, f_out, indent=2)

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total Row Count: 1020
Accepted Rows: 988
Rejected Rows: 32


Task 1.2 — Inspect and clean, pass 1

In [2]:
import pandas as pd

# 1. Load the original appointments file
df = pd.read_csv('appointments.csv')

# 2. Print initial dataset metrics
print("=== Initial Inspection ===")
print(f"Shape: {df.shape}")
print("\n--- Info ---")
df.info()
print("\n--- Missing Values ---")
print(df.isnull().sum())
print(f"\nDuplicate rows count: {df.duplicated().sum()}")
print("==========================\n")

# (a) Drop exact duplicate rows
df = df.drop_duplicates()
print(f"Row count after (a) dropping exact duplicates: {len(df)}")

# (b) Clean booking_channel with .str.strip().str.title()
df['booking_channel'] = df['booking_channel'].str.strip().str.title()
print(f"Row count after (b) cleaning booking_channel: {len(df)}")

# (c) Convert appt_date and booked_on to datetime
df['appt_date'] = pd.to_datetime(df['appt_date'], errors='coerce')
df['booked_on'] = pd.to_datetime(df['booked_on'], errors='coerce')
print(f"Row count after (c) converting dates: {len(df)}")

# (d) Drop rows where appt_date or status is missing
df = df.dropna(subset=['appt_date', 'status'])
print(f"Row count after (d) dropping rows with missing appt_date or status: {len(df)}")

=== Initial Inspection ===
Shape: (1020, 14)

--- Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

--- Missing Values ---
appointme

Task 1.3 — Clean, pass 2: validity, gaps and outliers

In [3]:
import numpy as np

# (a) Treat any age below 1 or above 100 as invalid and set it to missing
df.loc[(df['age'] < 1) | (df['age'] > 100), 'age'] = np.nan

# (b) Fill missing age with the median, then round and convert to int
age_median = df['age'].median()
df['age'] = df['age'].fillna(age_median).round().astype(int)

# Fill missing distance_km with its median
dist_median = df['distance_km'].median()
df['distance_km'] = df['distance_km'].fillna(dist_median)

# (c) Compute Q1, Q3, IQR and cap values at Q3 + 1.5*IQR and Q1 - 1.5*IQR
# For distance_km
q1_dist = df['distance_km'].quantile(0.25)
q3_dist = df['distance_km'].quantile(0.75)
iqr_dist = q3_dist - q1_dist
lower_dist = q1_dist - 1.5 * iqr_dist
upper_dist = q3_dist + 1.5 * iqr_dist

# For fee_inr
q1_fee = df['fee_inr'].quantile(0.25)
q3_fee = df['fee_inr'].quantile(0.75)
iqr_fee = q3_fee - q1_fee
lower_fee = q1_fee - 1.5 * iqr_fee
upper_fee = q3_fee + 1.5 * iqr_fee

# Count capped values
capped_dist_count = ((df['distance_km'] < lower_dist) | (df['distance_km'] > upper_dist)).sum()
capped_fee_count = ((df['fee_inr'] < lower_fee) | (df['fee_inr'] > upper_fee)).sum()

# Apply clipping
df['distance_km'] = np.clip(df['distance_km'], lower_dist, upper_dist)
df['fee_inr'] = np.clip(df['fee_inr'], lower_fee, upper_fee)

# (d) Round distance_km to 1 decimal and fee_inr to whole rupees
df['distance_km'] = df['distance_km'].round(1)
df['fee_inr'] = df['fee_inr'].round().astype(int)

print(f"Number of values capped in 'distance_km': {capped_dist_count}")
print(f"Number of values capped in 'fee_inr': {capped_fee_count}")

Number of values capped in 'distance_km': 26
Number of values capped in 'fee_inr': 3


Task 1.4 — Features and a NumPy summary

In [4]:
import pandas as pd
import requests
import time

# 1. Load clinics data
clinics_df = pd.read_csv('clinics.csv')
print("Original clinics data:")
display(clinics_df.head())

temp_c_list = []

for idx, row in clinics_df.iterrows():
    lat = row['latitude']
    lon = row['longitude']
    clinic_name = row['clinic_name']

    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    temp_c = None
    success = False

    # Attempt 1 & 2 (retry once)
    for attempt in range(2):
        try:
            response = requests.get("https://api.open-meteo.com/v1/forecast", params=params, timeout=10)
            if response.status_code == 200:
                data = response.json()
                if "hourly" in data and "temperature_2m" in data["hourly"] and len(data["hourly"]["temperature_2m"]) > 0:
                    temp_c = data["hourly"]["temperature_2m"][0]
                    success = True
                    break
                else:
                    reason = "hourly temperature_2m data missing in response"
            else:
                reason = f"HTTP Status Code {response.status_code}"
        except requests.exceptions.RequestException as e:
            reason = str(e)

        if not success and attempt == 0:
            print(f"Attempt 1 failed for {clinic_name}. Retrying...")
            time.sleep(1) # Small pause before retry

    if not success:
        print(f"Failed to get weather for clinic: {clinic_name}. Reason: {reason}")

    temp_c_list.append(temp_c)

# 2. Enrich and save to clinics_enriched.csv
clinics_df['temp_c'] = temp_c_list
clinics_df.to_csv('clinics_enriched.csv', index=False)
print("\nEnriched clinics data saved to clinics_enriched.csv!")
display(clinics_df)

Original clinics data:


,clinic_id,clinic_name,city,tier,latitude,longitude
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873



Enriched clinics data saved to clinics_enriched.csv!


,clinic_id,clinic_name,city,tier,latitude,longitude,temp_c
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,17.4
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,18.7
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882,19.9
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150,19.7
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873,19.7
5,C06,Edappally Medical Centre,Kochi,Tier-2,10.0261,76.3085,25.7
6,C07,Vijay Nagar Clinic,Indore,Tier-2,22.7533,75.8937,18.3
7,C08,Gomti Nagar Care Point,Lucknow,Tier-2,26.8467,80.9462,18.7


Task 1.6 — Export the clean table and write the SQL load script

In [5]:
import pandas as pd
import numpy as np

# 1. Compute missing features: lead_days and no_show
df['lead_days'] = (df['appt_date'] - df['booked_on']).dt.days
df['no_show'] = df['status'].apply(lambda x: 1 if x == 'No-Show' else 0)

# 2. Sort by appointment_id
df = df.sort_values(by='appointment_id')

# 3. Format date columns as YYYY-MM-DD text
df['booked_on'] = df['booked_on'].dt.strftime('%Y-%m-%d')
df['appt_date'] = df['appt_date'].dt.strftime('%Y-%m-%d')

# 4. Filter and order to the exact 16 columns
columns_order = [
    'appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date',
    'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type',
    'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show'
]
df_clean = df[columns_order]

# 5. Check constraints
is_unique = df_clean['appointment_id'].is_unique
missing_counts = df_clean.isnull().sum()

print(f"Is appointment_id unique? {is_unique}")
print("\nMissing values in the 16 clean columns:")
print(missing_counts)

# Save the cleaned dataframe
df_clean.to_csv('appointments_clean.csv', index=False)
print("\nSaved appointments_clean.csv successfully!")

# 6. Generate SQL Insert Statements
sql_lines = []

# Process clinics first
clinics_enriched = pd.read_csv('clinics_enriched.csv')
for _, row in clinics_enriched.iterrows():
    clinic_id = row['clinic_id']
    clinic_name = row['clinic_name']
    city = row['city']
    tier = row['tier']
    lat = row['latitude']
    lon = row['longitude']
    temp_c = row['temp_c']

    # Format temp_c for SQL (convert NaN to NULL, otherwise use unquoted numeric float)
    temp_val = 'NULL' if pd.isna(temp_c) else f"{temp_c}"

    stmt = f"INSERT INTO clinics VALUES ('{clinic_id}','{clinic_name}','{city}','{tier}',{lat},{lon},{temp_val});"
    sql_lines.append(stmt)

# Process appointments next
for _, row in df_clean.iterrows():
    # Format values for safe interpolation
    val_list = []
    for col in columns_order:
        val = row[col]
        if pd.isna(val):
            val_list.append("NULL")
        elif isinstance(val, (int, float, np.integer, np.floating)):
            # If it's a numeric type, print it unquoted
            val_list.append(str(val))
        else:
            # String/Date types, print single quoted
            val_list.append(f"'{val}'")

    stmt = f"INSERT INTO appointments VALUES ({','.join(val_list)});"
    sql_lines.append(stmt)

# Write SQL lines to load_data.sql
with open('load_data.sql', 'w', encoding='utf-8') as sql_file:
    sql_file.write('\n'.join(sql_lines) + '\n')

print(f"\nGenerated load_data.sql with {len(sql_lines)} total insert statements!")

Is appointment_id unique? True

Missing values in the 16 clean columns:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64

Saved appointments_clean.csv successfully!

Generated load_data.sql with 998 total insert statements!


In [6]:
import pandas as pd

# Load files
clinics = pd.read_csv("clinics_enriched.csv")
appointments = pd.read_csv("appointments_clean.csv")

# Equivalent of SQL JOIN
merged = appointments.merge(
    clinics,
    on="clinic_id",
    how="inner"
)

# Equivalent of GROUP BY
clinic_summary = (
    merged.groupby(["clinic_name", "city"], as_index=False)
          .agg(
              appointments=("appointment_id", "count"),
              no_shows=("no_show", "sum")
          )
)

# Equivalent of:
# ROUND(100.0 * SUM(no_show) / COUNT(*), 1)
clinic_summary["no_show_pct"] = (
    100 * clinic_summary["no_shows"]
    / clinic_summary["appointments"]
).round(1)

# Equivalent of HAVING COUNT(*) >= 95
result_95 = (
    clinic_summary[clinic_summary["appointments"] >= 95]
    .sort_values("no_show_pct", ascending=False)
)

print(result_95)

                clinic_name    city  appointments  no_shows  no_show_pct
2      Hadapsar Care Centre    Pune           143        39         27.3
4      Malviya Nagar Clinic  Jaipur           142        38         26.8
3     Kothrud Family Clinic    Pune           160        36         22.5
7        Vijay Nagar Clinic  Indore           101        21         20.8
0  Edappally Medical Centre   Kochi           142        23         16.2
6     Vaishali Nagar Clinic  Jaipur           130        18         13.8


In [7]:
all_clinics = (
    clinic_summary
    .sort_values("no_show_pct", ascending=False)
)

print(all_clinics)

                clinic_name     city  appointments  no_shows  no_show_pct
1    Gomti Nagar Care Point  Lucknow            90        30         33.3
2      Hadapsar Care Centre     Pune           143        39         27.3
4      Malviya Nagar Clinic   Jaipur           142        38         26.8
3     Kothrud Family Clinic     Pune           160        36         22.5
7        Vijay Nagar Clinic   Indore           101        21         20.8
0  Edappally Medical Centre    Kochi           142        23         16.2
5    Sitabuldi Health Point   Nagpur            82        13         15.9
6     Vaishali Nagar Clinic   Jaipur           130        18         13.8


In [8]:
highest = all_clinics.iloc[0]

print(f"Clinic       : {highest['clinic_name']}")
print(f"City         : {highest['city']}")
print(f"Appointments : {highest['appointments']}")
print(f"No Shows     : {highest['no_shows']}")
print(f"No Show %    : {highest['no_show_pct']}")

Clinic       : Gomti Nagar Care Point
City         : Lucknow
Appointments : 90
No Shows     : 30
No Show %    : 33.3


In [9]:
sql_like = (
    merged.groupby(["clinic_name", "city"])
          .agg(
              appointments=("appointment_id", "count"),
              no_shows=("no_show", "sum")
          )
          .reset_index()
)

print(sql_like)

                clinic_name     city  appointments  no_shows
0  Edappally Medical Centre    Kochi           142        23
1    Gomti Nagar Care Point  Lucknow            90        30
2      Hadapsar Care Centre     Pune           143        39
3     Kothrud Family Clinic     Pune           160        36
4      Malviya Nagar Clinic   Jaipur           142        38
5    Sitabuldi Health Point   Nagpur            82        13
6     Vaishali Nagar Clinic   Jaipur           130        18
7        Vijay Nagar Clinic   Indore           101        21
